---
jupyter: python3
---

# Team 4 live submission --- advanced setup {#sec-team4-submission}

::: {.callout-note title="Learning objective"}
You can render `team_4`'s signed-off live submission: a 24-hour DE load
forecast produced by a longer SpotOptim tuning run and a richer
feature/lag setup than the generic baseline in @sec-live-submission.
The chapter writes the validated CSV into the leaderboard fork and
documents the exact `git` / `gh` commands `team_4` uses to open the
submission PR.
:::

## Overview {#sec-team4-overview}

@sec-live-submission is a *generic* template: any team can render it,
substitute a placeholder, and submit. This chapter is **team_4's
tuned variant**. Every safety contract from chapter 13 carries over
(deadline check, 24-row schema, strictly-positive forecasts, fail-loud
ENTSO-E coverage assertion); only the modelling configuration changes.

What this chapter changes relative to @sec-live-submission:

- All five seasonal `Period`s instead of `daily` alone --- weekly,
  monthly, quarterly, and yearly RBF encodings join the daily one.
- Lag set extended from `[1, 2, 24]` to `[1, 2, 24, 168]` --- the
  weekly lag is a direct day-of-week persistence channel for a
  recursive forecaster.
- `include_weather_windows=True`, `include_holiday_features=True`,
  `include_poly_features=True` --- temperature, DE/NW public holidays,
  and bilinear calendar-by-weather interactions enter the exogenous
  matrix.
- SpotOptim budget raised from 10 / 5 to 30 / 10 (trials / initial
  evaluations), with `number_folds=10` --- roughly threefold more
  evaluations than the baseline.

## Submission contract

Unchanged --- see @sec-live-contract for the CSV schema, deadline, and
scoring rules. The leaderboard validator that runs against this
chapter's output is byte-identical to the one used by
@sec-live-submission.

## Team identity {#sec-team4-identity}

`team_4` is hardcoded throughout this chapter, there is no
placeholder to edit. The cell below confirms the team is registered in
`challenge-leaderboard/teams.yml`.

In [ ]:
#| label: team4-registry
#| echo: true
from pathlib import Path
import yaml

teams_yml = Path.home() / "workspace" / "challenge-leaderboard" / "teams.yml"
teams = yaml.safe_load(teams_yml.read_text())["teams"]
team4 = next((t for t in teams if t["id"] == "team_4"), None)
assert team4 is not None, "team_4 is not in challenge-leaderboard/teams.yml"
print(f"team_4 display_name : {team4['display_name']}")
print(f"team_4 github       : {', '.join(team4['github_handles'])}")

`validate_authorship()` in
`scripts/validate_submission.py:89` rejects PRs whose author is not in
this `github_handles` list, so the `gh auth login` account used in
@sec-team4-push must match.

## Prerequisites and walkthrough

Identical to @sec-live-walkthrough: Python 3.13 + uv, a personal
ENTSO-E API key exported as `ENTSOE_API_KEY`, `gh auth login` as a
registered `team_4` handle, and a sibling clone of
`challenge-leaderboard` at `~/workspace/challenge-leaderboard/`.

The only difference from the generic walkthrough is that
`TEAM_ID = "team_4"` is already baked into the submission cell at
@sec-team4-write --- nothing to edit.

## Live pipeline {#sec-team4-example}

The cells below are the *live* team_4 pipeline. Every render
downloads today's ENTSO-E actuals, tunes an LGBM forecaster with the
advanced configuration, and writes a validated CSV for tomorrow.

In [ ]:
#| label: team4-imports
#| echo: true
import logging, os
from pathlib import Path
import pandas as pd

logging.basicConfig(level=logging.WARNING)

from spotforecast2_safe.data import Period
from spotforecast2_safe.downloader.entsoe import download_new_data
from spotforecast2_safe.data.fetch_data import get_data_home
from spotforecast2_safe.configurator.config_entsoe import ConfigEntsoe
from spotforecast2.tasks.task_entsoe import (
    entsoe_data_loader,
    entsoe_test_data_loader,
    entsoe_lgbm_factory,
)
from spotforecast2.multitask import MultiTask

### Dynamic date constants

Every date is derived from `pd.Timestamp.now(tz="UTC").normalize()` so
the chapter stays correct on every render --- same convention as
@sec-live-example.

In [ ]:
#| label: team4-constants
#| echo: true
TODAY_UTC = pd.Timestamp.now(tz="UTC").normalize()
YESTERDAY_UTC = TODAY_UTC - pd.Timedelta(days=1)
TOMORROW_UTC = TODAY_UTC + pd.Timedelta(days=1)

START_DOWNLOAD = "202201010000"
END_DOWNLOAD = TODAY_UTC.strftime("%Y%m%d%H%M")

print(f"TODAY_UTC     = {TODAY_UTC}")
print(f"TOMORROW_UTC  = {TOMORROW_UTC}  (target day = y_0)")
print(f"END_DOWNLOAD  = {END_DOWNLOAD}")

### Download today's ENTSO-E actuals

In [ ]:
#| label: team4-download
#| echo: true
api_key = os.environ.get("ENTSOE_API_KEY")
assert api_key, "ENTSOE_API_KEY must be set in the environment."

download_new_data(
    api_key=api_key,
    country_code="DE",
    start=START_DOWNLOAD,
    end=END_DOWNLOAD,
    force=False,
)

interim_csv = get_data_home() / "interim" / "energy_load.csv"
interim = pd.read_csv(interim_csv, index_col=0, parse_dates=True)
interim.index = pd.to_datetime(interim.index, utc=True)

required_last = TODAY_UTC - pd.Timedelta(hours=1)
if interim.index.max() < required_last:
    raise RuntimeError(
        f"ENTSO-E coverage is stale: last interim row is {interim.index.max()} "
        f"but the chapter requires at least {required_last}. "
        "Re-run download_new_data(..., force=True) or check the ENTSO-E API."
    )
print(f"interim CSV last row: {interim.index.max()}")
print(interim["Actual Load"].dropna().tail(6))

### Training cutoff and forecast horizon

Identical mechanics to @sec-live-example: anchor `end_train` at the
last full hour of `Actual Load`, then forecast forward through
tomorrow 23:00 UTC and slice the 24 rows we submit.

In [ ]:
#| label: team4-cutoff
#| echo: true
LAST_TARGET = TOMORROW_UTC + pd.Timedelta(hours=23)
LAST_FULL_HOUR = interim["Actual Load"].dropna().index.max().floor("h")
FIRST_PRED = LAST_FULL_HOUR + pd.Timedelta(hours=1)
LIVE_N_STEPS = int((LAST_TARGET - FIRST_PRED).total_seconds() // 3600) + 1

print(f"end_train (inclusive): {LAST_FULL_HOUR}")
print(f"first forecast step  : {FIRST_PRED}")
print(f"last forecast step   : {LAST_TARGET}")
print(f"predict_size         : {LIVE_N_STEPS}  (slice last 24 for y_0)")

### Advanced `ConfigEntsoe` {#sec-team4-config}

Each change relative to @sec-live-config has a one-line rationale:

- All five seasonal periods --- captures multi-scale seasonality the
  recursive LGBM cannot recover from hourly lags alone.
- Weekly lag 168 --- direct day-of-week persistence channel.
- `include_weather_windows=True` --- temperature is the dominant load
  driver below ~5 °C and above ~15 °C.
- `include_holiday_features=True` --- DE/NW public holidays shift
  weekday load by 20--30 %.
- `include_poly_features=True` --- bilinear weather × calendar
  interactions; the larger tuning budget can absorb the wider feature
  space.
- `n_trials_spotoptim=30`, `n_initial_spotoptim=10`, `number_folds=10`
  --- roughly threefold more search evaluations than the baseline.

In [ ]:
#| label: team4-config
#| echo: true
TRAIN_SIZE = pd.Timedelta(days=365 * 3)
PREDICT_SIZE = 24
REFIT_SIZE = 7
NUMBER_FOLDS = 10
DELTA_VAL = pd.Timedelta(hours=PREDICT_SIZE * REFIT_SIZE * NUMBER_FOLDS)

ADVANCED_PERIODS = [
    Period(name="daily",     n_periods=12, column="hour",      input_range=(1, 24)),
    Period(name="weekly",    n_periods=7,  column="dayofweek", input_range=(0, 6)),
    Period(name="monthly",   n_periods=12, column="month",     input_range=(1, 12)),
    Period(name="quarterly", n_periods=4,  column="quarter",   input_range=(1, 4)),
    Period(name="yearly",    n_periods=12, column="dayofyear", input_range=(1, 365)),
]

ADVANCED_LAGS = [1, 2, 24, 168]

team4_config = ConfigEntsoe(
    country_code="DE",
    data_filename="interim/energy_load.csv",
    targets=["Actual Load"],
    agg_weights=[1.0],
    bounds=[(-1e9, 1e9)],
    data_loader=entsoe_data_loader,
    test_data_loader=entsoe_test_data_loader,
    forecaster_factory=entsoe_lgbm_factory,
    periods=ADVANCED_PERIODS,
    lags_consider=ADVANCED_LAGS,
    train_size=TRAIN_SIZE,
    end_train_default=LAST_FULL_HOUR.isoformat(),
    delta_val=DELTA_VAL,
    predict_size=LIVE_N_STEPS,
    refit_size=REFIT_SIZE,
    number_folds=NUMBER_FOLDS,
    n_trials_optuna=15,
    n_trials_spotoptim=30,
    n_initial_spotoptim=10,
    include_weather_windows=True,
    include_holiday_features=True,
    include_poly_features=True,
    state="NW",
    random_state=42,
    on_weather_failure="skip",
)
team4_config.data_frame_name = "ddmo-live-team4"
print("team4_config ready:", team4_config.data_frame_name,
      "| predict_size=", team4_config.predict_size,
      "| n_trials_spotoptim=", team4_config.n_trials_spotoptim,
      "| number_folds=", team4_config.number_folds)

### Train with SpotOptim and predict through tomorrow

The MultiTask orchestrator runs the full pipeline. With the advanced
configuration this cell typically takes 5--10 minutes on a recent
laptop --- mostly the 30 SpotOptim trials against 10 CV folds.

In [ ]:
#| label: team4-train
#| echo: true
team4_mt = MultiTask(team4_config, task="spotoptim")
team4_mt.prepare_data()
team4_mt.detect_outliers()
team4_mt.impute()
team4_mt.build_exogenous_features()
team4_mt.run(show=False)

future = team4_mt.results["spotoptim"]["Actual Load"]["future_pred"]
print(f"raw forecast: {len(future)} hourly steps "
      f"| {future.index.min()} → {future.index.max()}")

y0 = future.loc[TOMORROW_UTC:LAST_TARGET]
print(f"y_0 slice   : {len(y0)} hourly steps "
      f"| {y0.index.min()} → {y0.index.max()}")

### Visual inspection

In [ ]:
#| label: fig-team4-predict
#| echo: true
#| fig-cap: team_4 live ENTSO-E DE load forecast --- advanced setup (30 SpotOptim trials, 5 seasonal periods, weather windows, holiday and polynomial features). Training fit left of the dashed line; recursive forecast through tomorrow 23:00 UTC right of it. The 24 hourly values for `y_0` are the rightmost block.
from spotforecast2.plots.plotter import make_plot

pkg = team4_mt.results["spotoptim"]["Actual Load"]
print("future_pred :", len(pkg["future_pred"]), "rows,",
      pkg["future_pred"].index.min(), "→", pkg["future_pred"].index.max())
print("metrics_future         :", pkg.get("metrics_future"))
print("metrics_future_one_day :", pkg.get("metrics_future_one_day"))

fig = make_plot(pkg)
fig.show()

### Validate the schema and write the submission CSV {#sec-team4-write}

Same four contract assertions as @sec-live-example: 24 rows, first
stamp `TOMORROW 00:00 UTC`, every value strictly positive, no NaN. The
write target is `submissions/team_4/<tomorrow>.csv` inside the
leaderboard fork.

In [ ]:
#| label: team4-submission-write
#| echo: true
assert len(y0) == 24,           f"expected 24 hourly steps for y_0, got {len(y0)}"
assert y0.index.min() == TOMORROW_UTC, (
    f"first step {y0.index.min()} != TOMORROW {TOMORROW_UTC}"
)
assert (y0 > 0).all(),          "non-positive forecast value --- spec requires > 0"
assert y0.notna().all(),        "NaN in forecast --- spec forbids"

TEAM_ID = "team_4"
SUBMISSION_DIR = Path.home() / "workspace" / "challenge-leaderboard" / "submissions" / TEAM_ID
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
submission_path = SUBMISSION_DIR / f"{TOMORROW_UTC.date().isoformat()}.csv"

submission_df = pd.DataFrame({
    "timestamp_utc": y0.index.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "forecast_mw":   y0.round(2).values,
})
submission_df.to_csv(submission_path, index=False)
print(f"wrote {submission_path}")
print(f"  rows: {len(submission_df)}")
print(f"  range: {submission_df.forecast_mw.min():.1f}–{submission_df.forecast_mw.max():.1f} MW")
submission_df.head(3)

### Local validation with the leaderboard's own script

In [ ]:
#| label: team4-validate
#| echo: true
import subprocess
lb_repo = Path.home() / "workspace" / "challenge-leaderboard"
result = subprocess.run(
    [
        "uv", "run", "python", "scripts/validate_submission.py",
        "--path", str(submission_path.relative_to(lb_repo)),
        "--skip-deadline",
    ],
    cwd=lb_repo,
    capture_output=True,
    text=True,
)
print("exit code:", result.returncode)
print("stdout   :", result.stdout.strip())
if result.stderr.strip():
    print("stderr   :", result.stderr.strip())

## Commit and push the submission PR {#sec-team4-push}

The cells above only *write* the CSV --- they deliberately do not
commit, push, or open the PR (a careless render must not be able to
spam the leaderboard). Run the block below from a shell once the
validation cell above is green:

```bash
cd ~/workspace/challenge-leaderboard
TARGET=$(date -u -v+1d +%F)                          # macOS;
# TARGET=$(date -u -d 'tomorrow' +%F)                # Linux equivalent

git switch -c "submission/team_4-${TARGET}" main
git add "submissions/team_4/${TARGET}.csv"
git commit -m "submission(team_4): forecast for ${TARGET}"
git push -u origin "submission/team_4-${TARGET}"

gh pr create --base main --repo bartzbeielstein/challenge-leaderboard \
    --title "submission(team_4): forecast for ${TARGET}" \
    --body  "Generated by ddmo-sose-26 chapter 14 (team_4 advanced submission)."
```

If validation on the PR passes, auto-merge squashes the submission
onto `main`. The next morning at 07:00 UTC the leaderboard's
`score-daily.yml` cron (see @sec-live-cron) scores it against ENTSO-E
final actuals.

::: {.callout-warning title="Deadline"}
Submissions for target day `D` must be merged before `D-1 23:59 CET`
--- the validator enforces this strictly. The advanced configuration
in this chapter renders in 5--10 minutes on a typical laptop; give
yourself at least 30 minutes of head-room before the deadline.
:::

## What changed vs. chapter 13 --- summary {#sec-team4-summary}

| Knob | @sec-live-submission | This chapter |
|:--|:--|:--|
| `periods` | `daily` only | `daily, weekly, monthly, quarterly, yearly` |
| `lags_consider` | `[1, 2, 24]` | `[1, 2, 24, 168]` |
| `include_weather_windows` | (default) | `True` |
| `include_holiday_features` | (default) | `True` |
| `include_poly_features` | (default) | `True` |
| `n_trials_spotoptim` | `10` | `30` |
| `n_initial_spotoptim` | `5` | `10` |
| `number_folds` | `10` | `10` |
| Team id | `<team_id>` placeholder | `team_4` (hardcoded) |

::: {.callout-tip title="Exercises (with Claude)"}
1. *Did the bigger budget pay off?* Compare `metrics_future_one_day`
   from this chapter against the same metric in @sec-live-example for
   the same target day. Discuss with Claude whether the gain (or
   loss) justifies the extra ~5 minutes of compute.
2. *Ablate a feature.* Set `include_poly_features=False` and re-render
   only the train + write cells. Does the MAE move noticeably? If
   not, the interactions are not load-bearing for this dataset.
3. *Inspect the tuned hyperparameters.* Print
   `team4_mt.results["spotoptim"]["Actual Load"]` keys and locate the
   best-trial summary. Which lag configuration did SpotOptim pick?
:::